# Customer Churn and Revenue Retention Analysis

## Notebook 1: Data Cleaning and Exploratory Analysis

This notebook prepares the Telco Customer Churn dataset for business analysis, SQL reporting and churn modelling.

The aim is not only to predict churn, but to understand which customer groups are leaving, how much revenue is at risk, and where retention actions should be prioritised.

In [11]:
# Import required libraries

import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [12]:
# Load the dataset

file_path = "../data/telco_customer_churn.csv"

df = pd.read_csv(file_path)

# Preview the dataset
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Initial Dataset Inspection

This section checks the structure, size, data types and missing values in the dataset before cleaning.

In [13]:
# Check dataset shape

df.shape

(7043, 21)

In [14]:
# Check column names and data types

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [15]:
# Check missing values

missing_values = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_values[missing_values["missing_count"] > 0]

,missing_count,missing_percentage


In [16]:
# Check duplicate customer IDs

df["customerID"].duplicated().sum()

np.int64(0)

## 2. Data Cleaning

The initial inspection shows that `TotalCharges` is stored as text. This needs to be converted into a numeric field so it can be used for revenue-at-risk and customer value analysis.

In [17]:
# Check blank or non-numeric TotalCharges values

blank_total_charges = df[df["TotalCharges"].str.strip() == ""]

blank_total_charges.shape

(11, 21)

In [18]:
# Convert TotalCharges to numeric
# Blank spaces will become missing values after conversion

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Check missing values after conversion
df["TotalCharges"].isna().sum()

np.int64(11)

In [19]:
# Review rows where TotalCharges is missing after conversion

df[df["TotalCharges"].isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [20]:
# Replace missing TotalCharges with 0 for customers with zero tenure

df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0

# Confirm missing values have been fixed
df["TotalCharges"].isna().sum()

np.int64(0)

In [21]:
# Create a numeric churn flag for analysis and modelling

df["ChurnFlag"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

df[["Churn", "ChurnFlag"]].head()

,Churn,ChurnFlag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


In [22]:
# Convert SeniorCitizen into a readable category

df["SeniorCitizenLabel"] = df["SeniorCitizen"].map({
    1: "Senior",
    0: "Non-Senior"
})

df[["SeniorCitizen", "SeniorCitizenLabel"]].head()

,SeniorCitizen,SeniorCitizenLabel
0,0,Non-Senior
1,0,Non-Senior
2,0,Non-Senior
3,0,Non-Senior
4,0,Non-Senior


## 3. Business Feature Engineering

To make the analysis more useful for retention decision-making, additional business-focused fields are created. These include estimated annual revenue, customer value bands, tenure groups and churn risk interpretation fields.

In [23]:
# Create estimated annual revenue based on current monthly charges

df["EstimatedAnnualRevenue"] = df["MonthlyCharges"] * 12

# Create average monthly value based on total charges and tenure
# For zero-tenure customers, use MonthlyCharges as the current value proxy

df["AverageMonthlyValue"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

df[["MonthlyCharges", "TotalCharges", "tenure", "EstimatedAnnualRevenue", "AverageMonthlyValue"]].head()

,MonthlyCharges,TotalCharges,tenure,EstimatedAnnualRevenue,AverageMonthlyValue
0,29.85,29.85,1,358.20,29.85
1,56.95,"1,889.50",34,683.40,55.57
2,53.85,108.15,2,646.20,54.08
3,42.30,"1,840.75",45,507.60,40.91
4,70.70,151.65,2,848.40,75.83


In [24]:
# Create tenure groups for business analysis

def assign_tenure_group(tenure):
    if tenure == 0:
        return "New Customer"
    elif tenure <= 12:
        return "0-12 Months"
    elif tenure <= 24:
        return "13-24 Months"
    elif tenure <= 48:
        return "25-48 Months"
    else:
        return "49+ Months"

df["TenureGroup"] = df["tenure"].apply(assign_tenure_group)

df["TenureGroup"].value_counts().sort_index()

TenureGroup
0-12 Months     2175
13-24 Months    1024
25-48 Months    1594
49+ Months      2239
New Customer      11
Name: count, dtype: int64

In [25]:
# Create monthly charge bands

df["MonthlyChargeBand"] = pd.qcut(
    df["MonthlyCharges"],
    q=4,
    labels=["Low Value", "Mid-Low Value", "Mid-High Value", "High Value"]
)

df["MonthlyChargeBand"].value_counts()

MonthlyChargeBand
Mid-Low Value     1766
Low Value         1762
High Value        1758
Mid-High Value    1757
Name: count, dtype: int64

## 4. Overall Churn and Revenue-at-Risk Summary

This section calculates the overall churn rate and estimates the monthly and annual revenue currently at risk from customers who have churned.

In [26]:
# Overall customer and churn summary

total_customers = df["customerID"].nunique()
churned_customers = df["ChurnFlag"].sum()
active_customers = total_customers - churned_customers
overall_churn_rate = churned_customers / total_customers

summary_overview = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Active Customers",
        "Churned Customers",
        "Overall Churn Rate"
    ],
    "Value": [
        total_customers,
        active_customers,
        churned_customers,
        overall_churn_rate
    ]
})

summary_overview

,Metric,Value
0,Total Customers,"7,043.00"
1,Active Customers,"5,174.00"
2,Churned Customers,"1,869.00"
3,Overall Churn Rate,0.27


In [27]:
# Revenue-at-risk summary from churned customers

monthly_revenue_at_risk = df.loc[df["ChurnFlag"] == 1, "MonthlyCharges"].sum()
annual_revenue_at_risk = df.loc[df["ChurnFlag"] == 1, "EstimatedAnnualRevenue"].sum()
average_churned_customer_monthly_value = df.loc[df["ChurnFlag"] == 1, "MonthlyCharges"].mean()

revenue_risk_summary = pd.DataFrame({
    "Metric": [
        "Monthly Revenue at Risk",
        "Estimated Annual Revenue at Risk",
        "Average Monthly Value of Churned Customers"
    ],
    "Value": [
        monthly_revenue_at_risk,
        annual_revenue_at_risk,
        average_churned_customer_monthly_value
    ]
})

revenue_risk_summary

,Metric,Value
0,Monthly Revenue at Risk,"139,130.85"
1,Estimated Annual Revenue at Risk,"1,669,570.20"
2,Average Monthly Value of Churned Customers,74.44


In [28]:
# Create formatted versions of the summary tables for reporting

summary_overview_formatted = summary_overview.copy()
summary_overview_formatted["Value"] = summary_overview_formatted.apply(
    lambda row: f"{row['Value']:.1%}" if row["Metric"] == "Overall Churn Rate" else f"{row['Value']:,.0f}",
    axis=1
)

revenue_risk_summary_formatted = revenue_risk_summary.copy()
revenue_risk_summary_formatted["Value"] = revenue_risk_summary_formatted["Value"].apply(
    lambda x: f"£{x:,.2f}"
)

summary_overview_formatted, revenue_risk_summary_formatted

(               Metric  Value
 0     Total Customers  7,043
 1    Active Customers  5,174
 2   Churned Customers  1,869
 3  Overall Churn Rate  26.5%,
                                        Metric          Value
 0                     Monthly Revenue at Risk    £139,130.85
 1            Estimated Annual Revenue at Risk  £1,669,570.20
 2  Average Monthly Value of Churned Customers         £74.44)

### Initial Business Insight

The dataset contains 7,043 customers, of whom 1,869 have churned. This gives an overall churn rate of approximately 26.5%, meaning more than one in four customers have left the company.

From a revenue-retention perspective, churned customers represent around £139,130.85 in monthly recurring revenue and approximately £1.67m in estimated annual revenue at risk. The average churned customer has a monthly value of £74.44, which suggests that churn is not limited to low-value customers. This makes retention commercially important, not just operationally important.

## 5. Churn by Contract Type

Contract type is one of the most important churn drivers in subscription businesses. This section compares churn rate, customer count and revenue at risk across contract groups.

In [29]:
# Churn analysis by contract type

contract_churn = (
    df.groupby("Contract")
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
    .sort_values("ChurnRate", ascending=False)
)

contract_churn

,Contract,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
0,Month-to-month,3875,1655,0.43,"120,847.10","1,450,165.20"
1,One year,1473,166,0.11,"14,118.45","169,421.40"
2,Two year,1695,48,0.03,"4,165.30","49,983.60"


In [30]:
# Format contract churn table for reporting

contract_churn_formatted = contract_churn.copy()

contract_churn_formatted["ChurnRate"] = contract_churn_formatted["ChurnRate"].apply(lambda x: f"{x:.1%}")
contract_churn_formatted["MonthlyRevenueAtRisk"] = contract_churn_formatted["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
contract_churn_formatted["EstimatedAnnualRevenueAtRisk"] = contract_churn_formatted["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

contract_churn_formatted

,Contract,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
0,Month-to-month,3875,1655,42.7%,"£120,847.10","£1,450,165.20"
1,One year,1473,166,11.3%,"£14,118.45","£169,421.40"
2,Two year,1695,48,2.8%,"£4,165.30","£49,983.60"


### Contract Type Insight

Contract type shows a clear relationship with churn. Month-to-month customers have the highest churn rate at 42.7%, compared with 11.3% for one-year contracts and only 2.8% for two-year contracts.

This suggests that customers without a longer-term commitment are significantly more vulnerable to churn. From a revenue-retention perspective, month-to-month customers account for approximately £120,847 in monthly revenue at risk and around £1.45m in estimated annual revenue at risk. This makes the month-to-month segment the highest priority group for retention actions.

## 6. Churn by Tenure Group

Tenure analysis helps identify whether churn is concentrated among new customers, mid-tenure customers or long-term customers.

In [31]:
# Churn analysis by tenure group

tenure_churn = (
    df.groupby("TenureGroup")
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
)

# Apply custom order
tenure_order = ["New Customer", "0-12 Months", "13-24 Months", "25-48 Months", "49+ Months"]
tenure_churn["TenureGroup"] = pd.Categorical(
    tenure_churn["TenureGroup"],
    categories=tenure_order,
    ordered=True
)

tenure_churn = tenure_churn.sort_values("TenureGroup")

tenure_churn

,TenureGroup,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
4,New Customer,11,0,0.00,0.00,0.00
0,0-12 Months,2175,1037,0.48,"68,954.25","827,451.00"
1,13-24 Months,1024,294,0.29,"23,081.65","276,979.80"
2,25-48 Months,1594,325,0.20,"27,462.50","329,550.00"
3,49+ Months,2239,213,0.10,"19,632.45","235,589.40"


In [32]:
# Format tenure churn table for reporting

tenure_churn_formatted = tenure_churn.copy()

tenure_churn_formatted["ChurnRate"] = tenure_churn_formatted["ChurnRate"].apply(lambda x: f"{x:.1%}")
tenure_churn_formatted["MonthlyRevenueAtRisk"] = tenure_churn_formatted["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
tenure_churn_formatted["EstimatedAnnualRevenueAtRisk"] = tenure_churn_formatted["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

tenure_churn_formatted

,TenureGroup,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
4,New Customer,11,0,0.0%,£0.00,£0.00
0,0-12 Months,2175,1037,47.7%,"£68,954.25","£827,451.00"
1,13-24 Months,1024,294,28.7%,"£23,081.65","£276,979.80"
2,25-48 Months,1594,325,20.4%,"£27,462.50","£329,550.00"
3,49+ Months,2239,213,9.5%,"£19,632.45","£235,589.40"


### Tenure Group Insight

Tenure shows a strong relationship with churn. Customers in the first 12 months have the highest churn rate at 47.7%, while customers with 49 or more months of tenure have a much lower churn rate of 9.5%.

This suggests that the early customer experience is a major retention risk area. The first-year customer segment accounts for approximately £68,954 in monthly revenue at risk and around £827,451 in estimated annual revenue at risk. Retention activity should therefore focus strongly on onboarding, early support, service adoption and proactive engagement during the first 12 months.

## 7. Churn by Payment Method

Payment method can indicate differences in customer behaviour, friction and commitment. This section compares churn across payment types.

In [33]:
# Churn analysis by payment method

payment_churn = (
    df.groupby("PaymentMethod")
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
    .sort_values("ChurnRate", ascending=False)
)

payment_churn

,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
2,Electronic check,2365,1071,0.45,"84,288.75","1,011,465.00"
3,Mailed check,1612,308,0.19,"16,803.60","201,643.20"
0,Bank transfer (automatic),1544,258,0.17,"20,091.90","241,102.80"
1,Credit card (automatic),1522,232,0.15,"17,946.60","215,359.20"


In [34]:
# Format payment method churn table for reporting

payment_churn_formatted = payment_churn.copy()

payment_churn_formatted["ChurnRate"] = payment_churn_formatted["ChurnRate"].apply(lambda x: f"{x:.1%}")
payment_churn_formatted["MonthlyRevenueAtRisk"] = payment_churn_formatted["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
payment_churn_formatted["EstimatedAnnualRevenueAtRisk"] = payment_churn_formatted["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

payment_churn_formatted

,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
2,Electronic check,2365,1071,45.3%,"£84,288.75","£1,011,465.00"
3,Mailed check,1612,308,19.1%,"£16,803.60","£201,643.20"
0,Bank transfer (automatic),1544,258,16.7%,"£20,091.90","£241,102.80"
1,Credit card (automatic),1522,232,15.2%,"£17,946.60","£215,359.20"


### Payment Method Insight

Payment method shows a clear difference in churn behaviour. Customers using electronic check have the highest churn rate at 45.3%, compared with 16.7% for bank transfer and 15.2% for credit card automatic payments.

Electronic check customers also represent the largest revenue-retention risk, with approximately £84,289 in monthly revenue at risk and around £1.01m in estimated annual revenue at risk. This suggests that payment behaviour may be linked with customer commitment, billing friction or customer profile differences. A practical retention action would be to encourage high-risk electronic check customers to move toward automatic payment methods through incentives or improved billing communication.

## 8. Churn by Internet Service

Internet service type helps identify whether churn is linked to the type of service customers receive. This can support product, service quality and retention decisions.

In [35]:
# Churn analysis by internet service type

internet_churn = (
    df.groupby("InternetService")
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
    .sort_values("ChurnRate", ascending=False)
)

internet_churn

,InternetService,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
1,Fiber optic,3096,1297,0.42,"114,300.05","1,371,600.60"
0,DSL,2421,459,0.19,"22,529.20","270,350.40"
2,No,1526,113,0.07,"2,301.60","27,619.20"


In [36]:
# Format internet service churn table for reporting

internet_churn_formatted = internet_churn.copy()

internet_churn_formatted["ChurnRate"] = internet_churn_formatted["ChurnRate"].apply(lambda x: f"{x:.1%}")
internet_churn_formatted["MonthlyRevenueAtRisk"] = internet_churn_formatted["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
internet_churn_formatted["EstimatedAnnualRevenueAtRisk"] = internet_churn_formatted["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

internet_churn_formatted

,InternetService,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
1,Fiber optic,3096,1297,41.9%,"£114,300.05","£1,371,600.60"
0,DSL,2421,459,19.0%,"£22,529.20","£270,350.40"
2,No,1526,113,7.4%,"£2,301.60","£27,619.20"


### Internet Service Insight

Internet service type shows a major difference in churn behaviour. Fiber optic customers have the highest churn rate at 41.9%, compared with 19.0% for DSL customers and 7.4% for customers without internet service.

Fiber optic customers also represent the largest revenue-retention risk, with approximately £114,300 in monthly revenue at risk and around £1.37m in estimated annual revenue at risk. This suggests that fiber optic customers may require deeper investigation around service quality, pricing, expectations or competitor switching. Because this segment combines high churn with high revenue exposure, it should be treated as a priority retention group.

## 9. Churn by Monthly Charge Band

Monthly charge bands help assess whether churn is concentrated among lower-value or higher-value customers. This is important for prioritising retention actions based on commercial value.

In [37]:
# Churn analysis by monthly charge band

charge_band_churn = (
    df.groupby("MonthlyChargeBand", observed=False)
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
)

charge_band_churn

,MonthlyChargeBand,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
0,Low Value,1762,198,0.11,"4,775.20","57,302.40"
1,Mid-Low Value,1766,434,0.25,"23,970.25","287,643.00"
2,Mid-High Value,1757,659,0.38,"52,881.65","634,579.80"
3,High Value,1758,578,0.33,"57,503.75","690,045.00"


In [38]:
# Format monthly charge band churn table for reporting

charge_band_churn_formatted = charge_band_churn.copy()

charge_band_churn_formatted["ChurnRate"] = charge_band_churn_formatted["ChurnRate"].apply(lambda x: f"{x:.1%}")
charge_band_churn_formatted["MonthlyRevenueAtRisk"] = charge_band_churn_formatted["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
charge_band_churn_formatted["EstimatedAnnualRevenueAtRisk"] = charge_band_churn_formatted["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

charge_band_churn_formatted

,MonthlyChargeBand,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
0,Low Value,1762,198,11.2%,"£4,775.20","£57,302.40"
1,Mid-Low Value,1766,434,24.6%,"£23,970.25","£287,643.00"
2,Mid-High Value,1757,659,37.5%,"£52,881.65","£634,579.80"
3,High Value,1758,578,32.9%,"£57,503.75","£690,045.00"


### Monthly Charge Band Insight

Monthly charge bands show that churn is not concentrated only among low-value customers. The Mid-High Value group has the highest churn rate at 37.5%, while the High Value group carries the largest estimated annual revenue at risk at approximately £690,045.

This distinction is important for retention prioritisation. The business should not only focus on customer groups with the highest churn rate, but also on groups where churn creates the greatest revenue exposure. High-value and mid-high-value customers should therefore be prioritised for targeted retention offers, service reviews and proactive account engagement.

## 10. Combined High-Risk Segment Analysis

Single-variable churn analysis is useful, but retention decisions often require identifying customer groups where multiple risk factors overlap. This section combines contract type, tenure group, internet service and payment method to identify the highest-risk customer segments.

In [39]:
# Combined churn risk analysis by key customer segments

high_risk_segments = (
    df.groupby(["Contract", "TenureGroup", "InternetService", "PaymentMethod"], observed=False)
    .agg(
        Customers=("customerID", "nunique"),
        ChurnedCustomers=("ChurnFlag", "sum"),
        ChurnRate=("ChurnFlag", "mean"),
        MonthlyRevenueAtRisk=("MonthlyCharges", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum()),
        EstimatedAnnualRevenueAtRisk=("EstimatedAnnualRevenue", lambda x: x[df.loc[x.index, "ChurnFlag"] == 1].sum())
    )
    .reset_index()
)

# Keep only segments with enough customers to be meaningful
high_risk_segments = high_risk_segments[high_risk_segments["Customers"] >= 30]

# Sort by churn rate and annual revenue at risk
high_risk_segments = high_risk_segments.sort_values(
    ["ChurnRate", "EstimatedAnnualRevenueAtRisk"],
    ascending=[False, False]
)

high_risk_segments.head(10)

,Contract,TenureGroup,InternetService,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
4,Month-to-month,0-12 Months,Fiber optic,Bank transfer (automatic),87,71,0.82,"5,956.15","71,473.80"
6,Month-to-month,0-12 Months,Fiber optic,Electronic check,631,449,0.71,"37,166.00","445,992.00"
7,Month-to-month,0-12 Months,Fiber optic,Mailed check,123,77,0.63,"6,152.90","73,834.80"
5,Month-to-month,0-12 Months,Fiber optic,Credit card (automatic),75,46,0.61,"3,903.25","46,839.00"
18,Month-to-month,13-24 Months,Fiber optic,Electronic check,255,150,0.59,"13,353.65","160,243.80"
2,Month-to-month,0-12 Months,DSL,Electronic check,282,142,0.50,"6,318.90","75,826.80"
30,Month-to-month,25-48 Months,Fiber optic,Electronic check,286,140,0.49,"12,946.75","155,361.00"
17,Month-to-month,13-24 Months,Fiber optic,Credit card (automatic),62,28,0.45,"2,459.05","29,508.60"
28,Month-to-month,25-48 Months,Fiber optic,Bank transfer (automatic),102,42,0.41,"3,889.00","46,668.00"
3,Month-to-month,0-12 Months,DSL,Mailed check,255,102,0.40,"4,704.70","56,456.40"


In [40]:
# Format high-risk segment table for reporting

high_risk_segments_top10 = high_risk_segments.head(10).copy()

high_risk_segments_top10["ChurnRate"] = high_risk_segments_top10["ChurnRate"].apply(lambda x: f"{x:.1%}")
high_risk_segments_top10["MonthlyRevenueAtRisk"] = high_risk_segments_top10["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
high_risk_segments_top10["EstimatedAnnualRevenueAtRisk"] = high_risk_segments_top10["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

high_risk_segments_top10

,Contract,TenureGroup,InternetService,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk
4,Month-to-month,0-12 Months,Fiber optic,Bank transfer (automatic),87,71,81.6%,"£5,956.15","£71,473.80"
6,Month-to-month,0-12 Months,Fiber optic,Electronic check,631,449,71.2%,"£37,166.00","£445,992.00"
7,Month-to-month,0-12 Months,Fiber optic,Mailed check,123,77,62.6%,"£6,152.90","£73,834.80"
5,Month-to-month,0-12 Months,Fiber optic,Credit card (automatic),75,46,61.3%,"£3,903.25","£46,839.00"
18,Month-to-month,13-24 Months,Fiber optic,Electronic check,255,150,58.8%,"£13,353.65","£160,243.80"
2,Month-to-month,0-12 Months,DSL,Electronic check,282,142,50.4%,"£6,318.90","£75,826.80"
30,Month-to-month,25-48 Months,Fiber optic,Electronic check,286,140,49.0%,"£12,946.75","£155,361.00"
17,Month-to-month,13-24 Months,Fiber optic,Credit card (automatic),62,28,45.2%,"£2,459.05","£29,508.60"
28,Month-to-month,25-48 Months,Fiber optic,Bank transfer (automatic),102,42,41.2%,"£3,889.00","£46,668.00"
3,Month-to-month,0-12 Months,DSL,Mailed check,255,102,40.0%,"£4,704.70","£56,456.40"


### Combined High-Risk Segment Insight

The combined segment analysis shows that churn risk becomes much sharper when multiple risk factors overlap. The highest-risk groups are mainly month-to-month customers with short tenure and fiber optic service.

The segment with the highest churn rate is month-to-month, 0-12 month, fiber optic customers paying by bank transfer, with an 81.6% churn rate. However, from a business-priority perspective, the strongest retention target is the month-to-month, 0-12 month, fiber optic and electronic check segment. This group contains 631 customers, 449 churned customers, a 71.2% churn rate and approximately £445,992 in estimated annual revenue at risk.

This shows why churn analysis should not rely only on the highest churn percentage. Retention decisions should consider customer volume and revenue exposure alongside churn rate.

## 11. Retention Priority Scoring

To support business decision-making, a simple retention priority score is created using churn rate and estimated annual revenue at risk. This helps identify which customer groups should be prioritised first for retention actions.

In [41]:
# Create a retention priority score
# The score combines churn rate and annual revenue exposure

segment_priority = high_risk_segments.copy()

segment_priority["RevenueRiskRank"] = segment_priority["EstimatedAnnualRevenueAtRisk"].rank(
    method="dense",
    ascending=False
)

segment_priority["ChurnRateRank"] = segment_priority["ChurnRate"].rank(
    method="dense",
    ascending=False
)

# Lower rank number is better, so combine ranks and sort ascending
segment_priority["RetentionPriorityScore"] = (
    segment_priority["RevenueRiskRank"] + segment_priority["ChurnRateRank"]
)

segment_priority = segment_priority.sort_values(
    "RetentionPriorityScore",
    ascending=True
)

segment_priority.head(10)

,Contract,TenureGroup,InternetService,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,MonthlyRevenueAtRisk,EstimatedAnnualRevenueAtRisk,RevenueRiskRank,ChurnRateRank,RetentionPriorityScore
6,Month-to-month,0-12 Months,Fiber optic,Electronic check,631,449,0.71,"37,166.00","445,992.00",1.00,2.00,3.00
4,Month-to-month,0-12 Months,Fiber optic,Bank transfer (automatic),87,71,0.82,"5,956.15","71,473.80",6.00,1.00,7.00
18,Month-to-month,13-24 Months,Fiber optic,Electronic check,255,150,0.59,"13,353.65","160,243.80",2.00,5.00,7.00
7,Month-to-month,0-12 Months,Fiber optic,Mailed check,123,77,0.63,"6,152.90","73,834.80",5.00,3.00,8.00
30,Month-to-month,25-48 Months,Fiber optic,Electronic check,286,140,0.49,"12,946.75","155,361.00",3.00,7.00,10.00
2,Month-to-month,0-12 Months,DSL,Electronic check,282,142,0.50,"6,318.90","75,826.80",4.00,6.00,10.00
5,Month-to-month,0-12 Months,Fiber optic,Credit card (automatic),75,46,0.61,"3,903.25","46,839.00",9.00,4.00,13.00
3,Month-to-month,0-12 Months,DSL,Mailed check,255,102,0.40,"4,704.70","56,456.40",8.00,10.00,18.00
42,Month-to-month,49+ Months,Fiber optic,Electronic check,135,50,0.37,"4,815.10","57,781.20",7.00,12.00,19.00
28,Month-to-month,25-48 Months,Fiber optic,Bank transfer (automatic),102,42,0.41,"3,889.00","46,668.00",10.00,9.00,19.00


In [42]:
# Format retention priority table for reporting

segment_priority_top10 = segment_priority.head(10).copy()

segment_priority_top10["ChurnRate"] = segment_priority_top10["ChurnRate"].apply(lambda x: f"{x:.1%}")
segment_priority_top10["MonthlyRevenueAtRisk"] = segment_priority_top10["MonthlyRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")
segment_priority_top10["EstimatedAnnualRevenueAtRisk"] = segment_priority_top10["EstimatedAnnualRevenueAtRisk"].apply(lambda x: f"£{x:,.2f}")

segment_priority_top10[
    [
        "Contract",
        "TenureGroup",
        "InternetService",
        "PaymentMethod",
        "Customers",
        "ChurnedCustomers",
        "ChurnRate",
        "EstimatedAnnualRevenueAtRisk",
        "RetentionPriorityScore"
    ]
]

,Contract,TenureGroup,InternetService,PaymentMethod,Customers,ChurnedCustomers,ChurnRate,EstimatedAnnualRevenueAtRisk,RetentionPriorityScore
6,Month-to-month,0-12 Months,Fiber optic,Electronic check,631,449,71.2%,"£445,992.00",3.00
4,Month-to-month,0-12 Months,Fiber optic,Bank transfer (automatic),87,71,81.6%,"£71,473.80",7.00
18,Month-to-month,13-24 Months,Fiber optic,Electronic check,255,150,58.8%,"£160,243.80",7.00
7,Month-to-month,0-12 Months,Fiber optic,Mailed check,123,77,62.6%,"£73,834.80",8.00
30,Month-to-month,25-48 Months,Fiber optic,Electronic check,286,140,49.0%,"£155,361.00",10.00
2,Month-to-month,0-12 Months,DSL,Electronic check,282,142,50.4%,"£75,826.80",10.00
5,Month-to-month,0-12 Months,Fiber optic,Credit card (automatic),75,46,61.3%,"£46,839.00",13.00
3,Month-to-month,0-12 Months,DSL,Mailed check,255,102,40.0%,"£56,456.40",18.00
42,Month-to-month,49+ Months,Fiber optic,Electronic check,135,50,37.0%,"£57,781.20",19.00
28,Month-to-month,25-48 Months,Fiber optic,Bank transfer (automatic),102,42,41.2%,"£46,668.00",19.00


### Retention Priority Insight

The retention priority score confirms that the most commercially important segment is not simply the segment with the highest churn rate. The highest-priority group is month-to-month, 0-12 month, fiber optic customers paying by electronic check.

This group contains 631 customers, 449 churned customers, a churn rate of 71.2% and approximately £445,992 in estimated annual revenue at risk. It ranks first because it combines high churn probability with the largest revenue exposure.

This segment should be prioritised for retention actions such as onboarding support, first-year engagement, billing review, contract upgrade incentives and targeted service-quality follow-up.

## 12. Save Cleaned Dataset

The cleaned dataset is saved for later SQL analysis, modelling and Power BI reporting.

In [43]:
# Save cleaned dataset for later stages

cleaned_file_path = "../outputs/telco_customer_churn_cleaned.csv"

df.to_csv(cleaned_file_path, index=False)

cleaned_file_path

'../outputs/telco_customer_churn_cleaned.csv'

In [44]:
# Confirm final dataset shape and columns

df.shape, df.columns.tolist()

((7043, 27),
 ['customerID',
  'gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'MonthlyCharges',
  'TotalCharges',
  'Churn',
  'ChurnFlag',
  'SeniorCitizenLabel',
  'EstimatedAnnualRevenue',
  'AverageMonthlyValue',
  'TenureGroup',
  'MonthlyChargeBand'])